# 🔗 데이터 연결 설정 가이드

> **작성일**: 2026-06-30  
> **대상**: dt4_team1 전체 팀원

---

## 아키텍처 요약

```
Azure Blob Storage (dt4team1blob)
  └─ raw 컨테이너
      ├─ bird/       → Bronze (Delta Table)
      ├─ outbreak/   → Bronze (Delta Table)
      ├─ weather/    → Bronze (Delta Table)
      └─ disinfection/ → PostgreSQL 직접 적재
```

## 연결 구성 현황

| 구성요소 | 이름 | 설명 |
|---------|------|------|
| Storage Credential | `dt4_team1_blob_credential` | Access Connector 기반 Managed Identity |
| External Location | `dt4team1blob_raw` | `abfss://raw@dt4team1blob.dfs.core.windows.net/` |
| Secret Scope | `dt4_team1_secrets` | PostgreSQL 접속 정보 저장 |

In [ ]:
# ============================================
# 1. Secrets 등록 확인
# ============================================
# Secret 값은 보안상 절대 출력되지 않음 ([REDACTED])
# scope와 key 목록만 확인 가능

print("=== Secret Scopes ===")
for s in dbutils.secrets.listScopes():
    print(f"  {s.name}")

print("\n=== Keys in 'dt4_team1_secrets' ===")
for k in dbutils.secrets.list("dt4_team1_secrets"):
    print(f"  {k.key}")

In [ ]:
# ============================================
# 2. Azure Blob Storage 읽기
# ============================================
# External Location이 설정되어 있으므로 SAS 토큰/키 없이 바로 접근 가능
# Unity Catalog가 credential을 자동 관리

BASE_PATH = "abfss://raw@dt4team1blob.dfs.core.windows.net/raw"

# --- 폴더 구조 확인 ---
display(spark.sql(f"LIST '{BASE_PATH}/'"))

In [ ]:
# bird: batch 파티션 내 개별 JSON 파일
df_bird = spark.read.json(f"{BASE_PATH}/bird/")
print(f"bird rows: {df_bird.count()}, columns: {len(df_bird.columns)}")
display(df_bird.limit(5))

In [ ]:
# outbreak: 단일 JSON 파일
df_outbreak = spark.read.option("multiLine", "true").json(f"{BASE_PATH}/outbreak/kahis_hpai.json")
print(f"outbreak rows: {df_outbreak.count()}, columns: {len(df_outbreak.columns)}")
display(df_outbreak.limit(5))

In [ ]:
from pyspark.sql import functions as F
import re
from collections import Counter

weather_path = f"{BASE_PATH}/weather/dt=20260628/asos_daily.csv"
raw_text = spark.read.text(weather_path)

header_lines = [
    row['value']
    for row in raw_text.filter(F.col('value').startswith('#')).collect()
]
main_header = header_lines[2][1:]
sub_header = header_lines[3][1:]
unit_header = header_lines[4][1:]


def get_token_spans(text: str):
    result = []
    for m in re.finditer(r"\S+", text):
        result.append({
            'token': m.group(),
            'start': m.start(),
            'end': m.end(),
            'center': (m.start() + m.end()) / 2,
        })
    return result


def get_tokens_in_main_range(main_idx, main_tokens, child_tokens):
    current = main_tokens[main_idx]
    left = current['start']
    if main_idx < len(main_tokens) - 1:
        next_token = main_tokens[main_idx + 1]
        right = (current['center'] + next_token['center']) / 2
    else:
        right = float('inf')
    return [x['token'] for x in child_tokens if left <= x['center'] < right]


def sanitize_token(text: str):
    return re.sub(r"[^0-9A-Za-z가-힣%°/]+", "_", text).strip("_")

main_tokens = get_token_spans(main_header)
sub_tokens = get_token_spans(sub_header)
unit_tokens = get_token_spans(unit_header)

column_names = []
for i, main in enumerate(main_tokens):
    sub_parts = get_tokens_in_main_range(i, main_tokens, sub_tokens)
    unit_parts = get_tokens_in_main_range(i, main_tokens, unit_tokens)

    raw_name = "_".join([main['token']] + sub_parts).lower()
    unit_name = sanitize_token("_".join(unit_parts))
    final_name = f"{raw_name}({unit_name})" if unit_name else raw_name
    column_names.append(final_name)

counts = Counter(column_names)
# print('duplicates after unit suffix:', [c for c, n in counts.items() if n > 1])
# print('sample names:', column_names[38:51])

data_rows = (
    raw_text
    .filter(~F.col('value').startswith('#'))
    .filter(F.length(F.trim('value')) > 0)
    .withColumn('value', F.regexp_replace(F.trim('value'), r'\s+', ' '))
    .withColumn('cols', F.split(F.col('value'), ' '))
)

row_lengths = data_rows.select(F.size('cols').alias('n')).groupBy('n').count().orderBy('n')
# display(row_lengths)

weather_split_test = data_rows.filter(F.size('cols') == len(column_names)).select('cols')
df_weather_raw_test = weather_split_test.select(*[
    F.col('cols')[i].alias(column_names[i]) for i in range(len(column_names))
])

display(df_weather_raw_test.limit(10))

In [ ]:
# disinfection: 월별 파티션 내 JSON
df_disinfection = spark.read.option("multiLine", "true").json(f"{BASE_PATH}/disinfection/")
print(f"disinfection rows: {df_disinfection.count()}, columns: {len(df_disinfection.columns)}")
display(df_disinfection.limit(5))

In [ ]:
# ============================================
# 3. PostgreSQL 쓰기 (Secrets 방식)
# ============================================
# 코드에 credential이 노출되지 않음

# --- Secrets에서 접속정보 가져오기 ---
pg_host = dbutils.secrets.get("dt4_team1_secrets", "pg_host")
pg_port = dbutils.secrets.get("dt4_team1_secrets", "pg_port")
pg_database = dbutils.secrets.get("dt4_team1_secrets", "pg_database")
pg_user = dbutils.secrets.get("dt4_team1_secrets", "pg_user")
pg_password = dbutils.secrets.get("dt4_team1_secrets", "pg_password")

# --- 읽기 예제 ---
df_pg = spark.read.format("postgresql") \
    .option("host", pg_host) \
    .option("port", pg_port) \
    .option("database", pg_database) \
    .option("dbtable", "public.disinfection_facilities") \
    .option("user", pg_user) \
    .option("password", pg_password) \
    .load()

print(f"PostgreSQL disinfection_facilities rows: {df_pg.count()}")
display(df_pg.limit(5))

In [ ]:
# --- 쓰기 예제 ---
# disinfection 데이터를 PostgreSQL에 적재
# mode: overwrite(덮어쓰기), append(추가), ignore(존재시 무시)

# df_disinfection.write.format("postgresql") \
#     .option("host", pg_host) \
#     .option("port", pg_port) \
#     .option("database", pg_database) \
#     .option("dbtable", "public.disinfection_facilities") \
#     .option("user", pg_user) \
#     .option("password", pg_password) \
#     .mode("overwrite") \
#     .save()
#
# print("✓ PostgreSQL 쓰기 완료")

---

## 오늘 설정 작업 요약 (2026-06-30)

### 1️⃣ Azure Blob Storage → Databricks 연결

| 단계 | 내용 |
|------|------|
| Access Connector 생성 | Azure Portal에서 `dt4_team1_connector` 생성 |
| IAM 권한 부여 | 컨넥터의 Managed Identity에 **Storage Blob Data Contributor** 역할 할당 |
| Storage Credential | `dt4_team1_blob_credential` 생성 (Access Connector ID 연결) |
| External Location | `dt4team1blob_raw` → credential 변경 (`dt4_team1_databricks` → `dt4_team1_blob_credential`) |

**결과**: 모든 노트북에서 SAS 토큰 없이 `abfss://raw@dt4team1blob.dfs.core.windows.net/` 접근 가능

### 2️⃣ PostgreSQL 연결 (Secrets 방식)

| 단계 | 내용 |
|------|------|
| Secret Scope | `dt4_team1_secrets` 생성 |
| 저장된 키 | `pg_host`, `pg_port`, `pg_database`, `pg_user`, `pg_password` |
| 연결 대상 | `<PG_HOST>:<PG_PORT>/<PG_DATABASE>` |
| 스키마 | `public` |

**결과**: `dbutils.secrets.get()` 으로 credential 노출 없이 PostgreSQL 읽기/쓰기 가능

### ⚠️ 주의사항
- Secret 값은 UI/로그에 절대 노출되지 않음 (`[REDACTED]`)
- External Location 경유 접근 시 Unity Catalog 권한이 필요 (관리자에게 권한 요청)
- PostgreSQL IAM 역할: **Storage Blob Data Contributor** (✘ Table 아님, ✔ Blob)